# 04 — GTA + WITS EDA

Goal: bring in two tariff-specific datasets to complement the geopolitical signal from GDELT/ICEWS.

- **Global Trade Alert (GTA)** — discrete trade-policy interventions (tariff hikes, export curbs, subsidies). Used as a **label source** for tariff events affecting NVS corridors.
- **WITS (World Bank)** — historical applied tariff rates by reporter × partner × HS code. Used as a **continuous tariff trajectory** feature.

Both are public. Neither touches NVS internal data.

In [ ]:
# Cell 1: Imports

import os
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120

print('Imports successful.')

ModuleNotFoundError: No module named 'requests'

---
## Section B — WITS (World Bank Integrated Trade Solution)

### Acquisition
WITS exposes a free SDMX-JSON REST API. No registration. Endpoint:

```
http://wits.worldbank.org/API/V1/SDMX/V21/datasource/TRN/
  reporter/{rep}/partner/{par}/product/{prd}/year/{yr}/datatype/reported
```

Returns simple averages of MFN applied tariff rates. Annual frequency, so this is a slow-moving baseline feature — not for short-horizon prediction, but for **corridor exposure weighting**.

We'll pull NVS HS chapters (35, 23, 30) for the top NVS corridors, 2015–2022.

In [1]:
# Cell 7: WITS API wrapper
# Returns long-format DataFrame: reporter, partner, product, year, tariff_rate

WITS_BASE = 'http://wits.worldbank.org/API/V1/SDMX/V21/datasource/TRN'

def fetch_wits(reporter, partner, product, year_from=2015, year_to=2022, timeout=30):
    """Pull WITS MFN applied tariff. Uses CSV format for simplicity."""
    # WITS also supports CSV via a separate endpoint; SDMX-JSON is verbose.
    # Use the simpler 'reported' tariff CSV endpoint.
    url = (
        f'http://wits.worldbank.org/API/V1/SDMX/V21/datasource/TRN/'
        f'reporter/{reporter}/partner/{partner}/product/{product}/'
        f'year/{year_from}-{year_to}/datatype/reported?format=csv'
    )
    try:
        r = requests.get(url, timeout=timeout)
        r.raise_for_status()
    except Exception as e:
        print(f'  Failed: {reporter}-{partner}-{product}: {e}')
        return pd.DataFrame()

    from io import StringIO
    try:
        df = pd.read_csv(StringIO(r.text))
    except Exception as e:
        print(f'  Parse fail {reporter}-{partner}-{product}: {e}')
        return pd.DataFrame()

    # Normalise column names — WITS CSV varies
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    df['reporter_iso'] = reporter
    df['partner_iso'] = partner
    df['product_query'] = product
    return df

# Smoke test
smoke = fetch_wits('USA', 'CHN', '35', 2018, 2020)
print(f'Smoke test rows: {len(smoke)}')
if not smoke.empty:
    display(smoke.head())

  Failed: USA-CHN-35: name 'requests' is not defined


NameError: name 'pd' is not defined

In [ ]:
# Cell 8: Pull WITS for NVS corridors × NVS HS chapters

NVS_CORRIDORS = [
    ('USA', 'CHN'), ('CHN', 'USA'),
    ('USA', 'DEU'), ('DEU', 'USA'),
    ('USA', 'BRA'), ('BRA', 'USA'),
    ('CHN', 'IND'), ('IND', 'CHN'),
    ('DEU', 'CHN'), ('CHN', 'DEU'),
    ('FRA', 'CHN'), ('USA', 'IND'),
    ('USA', 'CAN'), ('CAN', 'USA'),
    ('DNK', 'USA'), ('DNK', 'CHN'),
]

NVS_HS = ['35', '23', '30']

all_wits = []
for rep, par in NVS_CORRIDORS:
    for hs in NVS_HS:
        print(f'Fetching {rep}-{par} HS{hs}...')
        df = fetch_wits(rep, par, hs, 2015, 2022)
        if not df.empty:
            all_wits.append(df)

if all_wits:
    wits = pd.concat(all_wits, ignore_index=True)
    print()
    print(f'Combined shape: {wits.shape}')
    print(f'Columns: {list(wits.columns)}')
    display(wits.head())
else:
    print('No WITS data returned. Check network access from this machine, or use')
    print('the WITS bulk-download UI at https://wits.worldbank.org/ and load manually.')
    wits = pd.DataFrame()

In [ ]:
# Cell 9: WITS — tariff trajectories per corridor
# Identify the actual rate column dynamically — WITS naming inconsistent

if wits.empty:
    print('Skipping plot — no WITS data.')
else:
    # Find a numeric column that looks like a tariff rate
    rate_candidates = [
        c for c in wits.columns
        if any(k in c for k in ('tariff', 'rate', 'ahs', 'mfn', 'simple_average', 'value'))
    ]
    print('Rate-like columns:', rate_candidates)

    year_col = next((c for c in wits.columns if c in ('year', 'time_period', 'reference_period')), None)
    rate_col = rate_candidates[0] if rate_candidates else None

    if rate_col and year_col:
        plot_df = wits.copy()
        plot_df['year'] = pd.to_numeric(plot_df[year_col], errors='coerce')
        plot_df['rate'] = pd.to_numeric(plot_df[rate_col], errors='coerce')
        plot_df['corridor'] = plot_df['reporter_iso'] + '→' + plot_df['partner_iso']
        plot_df['corridor_hs'] = plot_df['corridor'] + ' (HS' + plot_df['product_query'].astype(str) + ')'

        # Aggregate in case multiple rows per corridor/year
        agg = plot_df.groupby(['corridor_hs', 'year'])['rate'].mean().reset_index()

        fig, ax = plt.subplots(figsize=(14, 7))
        for name, sub in agg.groupby('corridor_hs'):
            ax.plot(sub['year'], sub['rate'], marker='o', label=name, alpha=0.7, linewidth=1.4)
        ax.set_title('WITS applied tariffs — NVS corridors × HS chapters, 2015–2022', fontweight='bold')
        ax.set_ylabel('Tariff rate (%)')
        ax.set_xlabel('Year')
        ax.legend(fontsize=7, ncol=2, loc='upper left', bbox_to_anchor=(1.01, 1))
        plt.tight_layout()
        plt.savefig('../outputs/wits_tariff_trajectories.png', dpi=120, bbox_inches='tight')
        plt.show()
        print('Saved: outputs/wits_tariff_trajectories.png')

        wits.to_csv('../outputs/wits_nvs_corridors.csv', index=False)
        print('Saved: outputs/wits_nvs_corridors.csv')
    else:
        print('Could not locate year/rate columns automatically — inspect wits.columns and adjust.')

In [ ]:
# Cell 10: Cross-check — do GTA events line up with WITS rate jumps?
# Sanity check that both datasets agree on the major US-China 2018 escalation

if wits.empty or gta.empty or g_nvs.empty:
    print('Skipping — need both GTA and WITS loaded.')
else:
    # WITS USA→CHN HS35 trajectory
    usa_chn = wits[(wits['reporter_iso'] == 'USA') & (wits['partner_iso'] == 'CHN')]
    if not usa_chn.empty and rate_col and year_col:
        usa_chn = usa_chn.copy()
        usa_chn['year'] = pd.to_numeric(usa_chn[year_col], errors='coerce')
        usa_chn['rate'] = pd.to_numeric(usa_chn[rate_col], errors='coerce')
        usa_chn_agg = usa_chn.groupby('year')['rate'].mean().reset_index()

        # GTA: US-implemented tariffs hitting China
        gta_us_chn = gta_pairs[
            (gta_pairs['implementer'].str.contains('United States', na=False)) &
            (gta_pairs['affected'] == 'China')
        ].copy()
        gta_us_chn['year'] = gta_us_chn['date'].dt.year
        gta_us_chn_yearly = gta_us_chn.groupby('year').size().reset_index(name='gta_event_count')

        fig, ax1 = plt.subplots(figsize=(14, 5))
        ax1.plot(usa_chn_agg['year'], usa_chn_agg['rate'],
                 color='steelblue', marker='o', linewidth=2, label='WITS applied tariff (US→CHN, avg HS)')
        ax1.set_ylabel('Applied tariff rate (%)', color='steelblue')
        ax1.set_xlabel('Year')

        ax2 = ax1.twinx()
        ax2.bar(gta_us_chn_yearly['year'], gta_us_chn_yearly['gta_event_count'],
                alpha=0.3, color='coral', label='GTA tariff events (US→CHN)')
        ax2.set_ylabel('GTA tariff event count', color='coral')

        ax1.set_title('Cross-check: WITS tariff rate vs GTA event count — US→China',
                      fontweight='bold')
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')
        plt.tight_layout()
        plt.savefig('../outputs/gta_wits_cross_check.png', dpi=120, bbox_inches='tight')
        plt.show()
        print('Saved: outputs/gta_wits_cross_check.png')

---
## Summary checklist

- [ ] GTA CSVs dropped in `../data/gta/`
- [ ] GTA filtered to NVS corridors + tariff interventions → `gta_corridor_events.csv`
- [ ] WITS API reachable → `wits_nvs_corridors.csv`
- [ ] US→China spike (2018–2019) visible in **both** GTA event count and WITS rate
- [ ] Top corridors by GTA event count align with NVS internal corridor ranking (to be cross-referenced once internal data is joined)

### What feeds into the model
- **GTA** → binary/count label per (corridor, week): `tariff_event_t`. This is the **target**.
- **WITS** → continuous corridor-exposure baseline: `applied_tariff_t`. This is a **feature** (or used to weight predictions by actual exposure).
- **GDELT/ICEWS** features remain the leading geopolitical indicators.